# RHI Predictor Test

Uses the trained models and the RHI results in the project `outputs/` folder.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'models').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / 'models'
DATA_DIR = PROJECT_ROOT / 'data'
RHI_OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TEST_OUTPUT_DIR = PROJECT_ROOT / 'outputs_test'
TEST_OUTPUT_DIR.mkdir(exist_ok=True)

model1 = joblib.load(MODEL_DIR / 'iri_prediction_model.pkl')
kmeans_fwd = joblib.load(MODEL_DIR / 'fwd_kmeans_model.pkl')
scaler_fwd = joblib.load(MODEL_DIR / 'fwd_scaler.pkl')
le_pav = joblib.load(MODEL_DIR / 'fwd_le_pav.pkl')
le_lane = joblib.load(MODEL_DIR / 'fwd_le_lane.pkl')
health_mapping = joblib.load(MODEL_DIR / 'fwd_health_mapping.pkl')

print('Models loaded successfully.')

In [ ]:
print('Pavement families:', list(le_pav.classes_))
print('Lane types:', list(le_lane.classes_))

In [ ]:
# UPDATED: Added climate features to features1
features1 = ['MRI', 'AADTT_ALL_TRUCKS_TREND', 'ANNUAL_TRUCK_VOLUME_TREND',
             'ANNUAL_ESAL_TREND', 'CUMULATIVE_ESAL', 'YEAR',
             'MEAN_ANN_TEMP_AVG', 'FREEZE_INDEX_YR', 'FREEZE_THAW_YR']

features2 = ['PEAK_DEFL_1', 'PEAK_DEFL_2', 'PEAK_DEFL_3', 'PEAK_DEFL_4',
             'PEAK_DEFL_5', 'PEAK_DEFL_6', 'PEAK_DEFL_7', 'DROP_LOAD',
             'DROP_HEIGHT', 'PAVEMENT_FAMILY_ENC', 'LANE_NO_ENC']

def predict_rhi(sample):
    """Return IRI, FWD, and combined RHI predictions for one road sample."""
    unknown_pavement = set([sample['pavement_family']]) - set(le_pav.classes_)
    unknown_lane = set([sample['lane_no']]) - set(le_lane.classes_)
    if unknown_pavement or unknown_lane:
        raise ValueError(f'Unsupported category: pavement={unknown_pavement}, lane={unknown_lane}')

    # UPDATED: Added climate data mappings to the DataFrame
    iri_input = pd.DataFrame([[
        sample['mri'], sample['aadtt'], sample['annual_truck_volume'],
        sample['annual_esal'], sample['cumulative_esal'], sample['year'],
        sample['mean_temp'], sample['freeze_index'], sample['freeze_thaw']
    ]], columns=features1)
    
    predicted_iri = float(model1.predict(iri_input)[0])
    failure_threshold = 2.5
    iri_score = float(np.clip(
        ((failure_threshold - predicted_iri) / failure_threshold) * 100, 0, 100
    ))

    fwd_input = pd.DataFrame([[
        *sample['deflections'], sample['drop_load'], sample['drop_height'],
        le_pav.transform([sample['pavement_family']])[0],
        le_lane.transform([sample['lane_no']])[0]
    ]], columns=features2)
    
    fwd_scaled = scaler_fwd.transform(fwd_input)
    cluster = kmeans_fwd.predict(fwd_scaled)[0]
    health = health_mapping[cluster]
    
    # Reverse mapping to find exact indices for Good and Poor clusters
    reverse_mapping = {v: k for k, v in health_mapping.items()}
    good_cluster_idx = reverse_mapping['Good']
    poor_cluster_idx = reverse_mapping['Poor']
    
    # Calculate EXACT continuous structural health score using cluster distances
    distances = kmeans_fwd.transform(fwd_scaled)
    dist_to_good = float(distances[0, good_cluster_idx])
    dist_to_poor = float(distances[0, poor_cluster_idx])
    
    # Apply continuous formula
    fwd_score = (dist_to_poor / (dist_to_good + dist_to_poor)) * 100

    rhi = (iri_score + fwd_score) / 2
    condition = 'Good' if rhi >= 75 else 'Fair' if rhi >= 50 else 'Poor'
    return {
        'predicted_future_iri': predicted_iri,
        'iri_score': iri_score, 'fwd_health': health, 'fwd_score': fwd_score,
        'rhi': rhi, 'road_condition': condition
    }

In [ ]:
# UPDATED: Use multiple test points and pick one at random for each run
test_samples = [
    {
        'mri': 0.85,
        'aadtt': 950,
        'annual_truck_volume': 346750,
        'annual_esal': 310000,
        'cumulative_esal': 1500000,
        'year': 2025,
        'mean_temp': 15.5,
        'freeze_index': 10,
        'freeze_thaw': 45,
        'deflections': [450, 280, 210, 180, 140, 110, 70],
        'drop_load': 710,
        'drop_height': 4,
        'pavement_family': 'ACUB',
        'lane_no': 'F3',
    },
    {
        'mri': 1.20,
        'aadtt': 1120,
        'annual_truck_volume': 420000,
        'annual_esal': 390000,
        'cumulative_esal': 1750000,
        'year': 2026,
        'mean_temp': 13.0,
        'freeze_index': 25,
        'freeze_thaw': 60,
        'deflections': [520, 330, 240, 190, 160, 120, 85],
        'drop_load': 740,
        'drop_height': 4,
        'pavement_family': 'ACSP',
        'lane_no': 'F2',
    },
    {
        'mri': 0.65,
        'aadtt': 820,
        'annual_truck_volume': 285000,
        'annual_esal': 240000,
        'cumulative_esal': 980000,
        'year': 2024,
        'mean_temp': 18.0,
        'freeze_index': 5,
        'freeze_thaw': 18,
        'deflections': [390, 250, 170, 135, 110, 85, 55],
        'drop_load': 690,
        'drop_height': 4,
        'pavement_family': 'ACBF',
        'lane_no': 'F3',
    },
    {
        'mri': 1.05,
        'aadtt': 1400,
        'annual_truck_volume': 520000,
        'annual_esal': 460000,
        'cumulative_esal': 2050000,
        'year': 2027,
        'mean_temp': 11.0,
        'freeze_index': 45,
        'freeze_thaw': 90,
        'deflections': [600, 410, 310, 250, 200, 155, 110],
        'drop_load': 760,
        'drop_height': 4,
        'pavement_family': 'ACUB',
        'lane_no': 'F1',
    },
    {
        'mri': 0.95,
        'aadtt': 1030,
        'annual_truck_volume': 365000,
        'annual_esal': 335000,
        'cumulative_esal': 1580000,
        'year': 2025,
        'mean_temp': 16.5,
        'freeze_index': 15,
        'freeze_thaw': 55,
        'deflections': [475, 305, 235, 185, 150, 115, 75],
        'drop_load': 725,
        'drop_height': 4,
        'pavement_family': 'ACSP',
        'lane_no': 'F2',
    },
]

sample = test_samples[np.random.choice(len(test_samples))]
print('Selected random test sample:')
print(sample)

sample_result = pd.DataFrame([predict_rhi(sample)])
sample_result.to_csv(TEST_OUTPUT_DIR / 'sample_prediction.csv', index=False)
display(sample_result.round(2))
print(f'Saved sample prediction to {TEST_OUTPUT_DIR / "sample_prediction.csv"}')